# 01 - 数据探索: PLAsTiCC 天文光变曲线数据集

**目标**: 理解 PLAsTiCC 数据集的结构, 可视化不同类型天体的光变曲线特征。

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100
%matplotlib inline

## 1. 下载并加载数据

In [ ]:
from src.data.download import download_plasticc, load_plasticc

# 首次运行会从 HuggingFace 下载 (~300MB 压缩包)
# download_plasticc("../data/raw/plasticc")

# 加载数据
train, test, meta = load_plasticc("../data/raw/plasticc")

In [ ]:
print(f"训练集形状: {train.shape}")
print(f"测试集形状: {test.shape}")
print(f"\n列名: {list(train.columns)}")
print(f"\n前 5 行:")
train.head()

## 2. 数据概览

PLAsTiCC 数据集的每一行代表一个天体的一次观测。关键列:
- `object_id`: 天体唯一标识
- `mjd`: 观测时间 (Modified Julian Date)
- `flux`: 流量 (亮度)
- `flux_err`: 流量误差
- `passband`: 观测波段 (0-5, 对应 ugrizy 六个滤镜)
- `detected`: 是否被检测到 (0/1)
- `target`: 天体类别标签 (0-14, 但 PLAsTiCC 实际有 18 类)

In [ ]:
from src.data.preprocess import CLASS_NAMES

# 统计每个类别的天体数量
if meta is not None:
    if "target" in meta.columns:
        class_counts = meta["target"].value_counts().sort_index()
        for cls_id, count in class_counts.items():
            name = CLASS_NAMES.get(cls_id, f"unknown_{cls_id}")
            print(f"  {cls_id:4d}  {name:40s}  {count:6d}")

        # 柱状图
        fig, ax = plt.subplots(figsize=(14, 5))
        names = [CLASS_NAMES.get(c, str(c)) for c in class_counts.index]
        ax.bar(range(len(names)), class_counts.values, color="steelblue")
        ax.set_xticks(range(len(names)))
        ax.set_xticklabels(names, rotation=45, ha="right", fontsize=8)
        ax.set_ylabel("Number of objects")
        ax.set_title("PLAsTiCC 数据集: 各类天体数量")
        plt.tight_layout()
        plt.show()

## 3. 单条光变曲线示例

从一个天体提取光变曲线并绘制。

In [ ]:
from src.utils.visualization import plot_light_curve

# 取第一个 object_id
sample_id = train["object_id"].iloc[0]
sample = train[train["object_id"] == sample_id].sort_values("mjd")

print(f"Object ID: {sample_id}")
print(f"观测点数: {len(sample)}")
print(f"时间跨度: {sample['mjd'].max() - sample['mjd'].min():.1f} 天")
print(f"波段: {sorted(sample['passband'].unique())}")

# 按波段分别绘制
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 左图: 所有波段叠加
for pb in sorted(sample["passband"].unique()):
    mask = sample["passband"] == pb
    axes[0].errorbar(sample.loc[mask, "mjd"], sample.loc[mask, "flux"],
                     yerr=sample.loc[mask, "flux_err"], fmt="o",
                     markersize=3, alpha=0.7, label=f"band {pb}")
axes[0].set_xlabel("MJD")
axes[0].set_ylabel("Flux")
axes[0].set_title(f"Object {sample_id} — All bands")
axes[0].legend(fontsize=7)
axes[0].invert_yaxis()

# 右图: 单独波段 (passband=3, r-band)
band = sample[sample["passband"] == 3]
plot_light_curve(band["mjd"], band["flux"], band["flux_err"],
                 title=f"Object {sample_id} — r-band only", ax=axes[1])

plt.tight_layout()
plt.show()

## 4. 各类天体的光变曲线对比

每种天体类型的亮度变化模式不同。

In [ ]:
# 选取几个典型的类别进行对比
target_classes = {
    90: "SNIa (超新星 Ia 型)",
    42: "SNII (超新星 II 型)",
    62: "SNIbc (超新星 Ibc 型)",
    95: "SLSN-I (超亮超新星)",
    15: "TDE (潮汐瓦解事件)",
    88: "AGN (活动星系核)",
    16: "Eclipsing Binary (食双星)",
    92: "RR Lyrae (天琴座RR型变星)",
    53: "Mira (米拉变星)",
}

if meta is not None:
    fig, axes = plt.subplots(3, 3, figsize=(16, 12))
    axes = axes.flatten()

    for i, (cls_id, cls_name) in enumerate(target_classes.items()):
        ax = axes[i]
        # 找该类别的一个示例天体
        obj_ids = meta[meta["target"] == cls_id]["object_id"].values
        if len(obj_ids) == 0:
            ax.set_title(f"{cls_id}: {cls_name} (无数据)")
            continue

        obj_id = obj_ids[0]
        obj_data = train[train["object_id"] == obj_id].sort_values("mjd")
        band_data = obj_data[obj_data["passband"] == 3]  # r-band

        if len(band_data) > 0:
            ax.errorbar(band_data["mjd"], band_data["flux"],
                        yerr=band_data["flux_err"], fmt="o",
                        markersize=2, alpha=0.7, color=f"C{i}")
            ax.invert_yaxis()
        ax.set_title(f"{cls_id}: {cls_name}", fontsize=9)
        ax.set_xlabel("MJD")
        ax.set_ylabel("Flux")

    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle("各类天体 r-band 光变曲线示例", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

## 5. 关键特征分布

提取统计特征, 观察不同类别之间的差异。

In [ ]:
from src.features.extractor import extract_features
from tqdm.notebook import tqdm

if meta is not None:
    # 每个类别取 50 个样本, 提取特征
    sample_objects = []
    for cls_id in target_classes:
        obj_ids = meta[meta["target"] == cls_id]["object_id"].values[:50]
        for oid in obj_ids:
            sample_objects.append((oid, cls_id))

    print(f"共 {len(sample_objects)} 个样本")

    features_list = []
    labels = []
    for oid, cls_id in tqdm(sample_objects, desc="提取特征"):
        obj = train[train["object_id"] == oid]
        band = obj[obj["passband"] == 3]
        if len(band) >= 3:
            feat = extract_features(
                band["mjd"].values,
                band["flux"].values,
                band["flux_err"].values
            )
            features_list.append(feat)
            labels.append(cls_id)

    feat_df = pd.DataFrame(features_list)
    feat_df["target"] = labels
    print(f"特征矩阵: {feat_df.shape}")

In [ ]:
# 画几个关键特征的箱线图
key_features = ["flux_std", "flux_range", "amplitude_90", "autocorr_lag1", "snr_max"]
feat_labels = ["Flux Std", "Flux Range", "Amplitude (P5-P95)", "Autocorr Lag1", "SNR Max"]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, (feat, label) in enumerate(zip(key_features, feat_labels)):
    ax = axes[i]
    data_by_class = [feat_df[feat_df["target"] == c][feat].dropna().values
                     for c in target_classes.keys()]
    bp = ax.boxplot(data_by_class, labels=list(target_classes.values()),
                    patch_artist=True, vert=False)
    for patch, j in zip(bp["boxes"], range(len(target_classes))):
        patch.set_facecolor(f"C{j}")
    ax.set_title(label, fontsize=11)
    ax.tick_params(labelsize=7)

axes[-1].set_visible(False)
plt.suptitle("各类天体特征分布对比", fontsize=14)
plt.tight_layout()
plt.show()

## 6. 下一步

- **02_feature_engineering**: 批量提取特征, 构建训练/验证/测试集
- **03_baseline_model**: 用 XGBoost 训练基线分类器
- **04_lstm_model**: 用 LSTM 直接从光变曲线序列学习
- **05_human_review**: 找出模型不确定的样本, 人工校正